In [1]:
!pip install langchain  langgraph  langchain-community

In [2]:
!pip install -U "langchain[huggingface]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 35.2 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
  Attempting uninstall: langgraph-prebuilt━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [langchain-core]
    Found existing installation: langgraph-prebuilt 1.0.10━━━━ 1/5 [langchain-core]
    Uninstalling langgraph-prebuilt-1.0.10:━━━━━━━━━━━━━━━━━━━ 1/5 [langchain-core]
      Successfully uninstalled langgraph-prebuilt-1.0.10━━━━━━ 1/5 [langchain-core]
  Attempting uninstall: langgraph━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [langchain-core]
    Found existing installation: langgraph 1.1.9━━━━━━━━━━━━━━ 1/5 [langchain-core]
    Uninstalling langgraph-1.1.9:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [langchain-core]
      Successfully uninstalled langgraph-1.1.9━━━━━━━━━━━━━━━━ 1/5 [langchain-core]
  Attempting uninstall: langchain╺━━━━━━━━━━━━━━━ 3/5 [langgra

In [5]:
!pip install -U langchain-community

In [6]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_uYdepMBccfXSDluLwKNtItciIHa"

from langchain_community.chat_models import ChatOllama

model = ChatOllama(
    model="microsoft/Phi-3-mini-4k-instruct",
    temperature=0.7,
    max_tokens=1024,
)

/tmp/ipykernel_214980/2846547774.py:6: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  model = ChatOllama(


In [7]:
import requests, pathlib

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

Chinook.db already exists, skipping download.


In [8]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

Dialect: sqlite
Available tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [9]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [15]:
# Create a simple SQL agent (works without Ollama)
def ask_sql_agent(question: str) -> str:
    """Ask a question to the SQL agent and execute SQL queries."""
    
    # Create tool map for easy access
    tool_map = {tool.name: tool for tool in tools}
    query_tool = tool_map.get("sql_db_query")
    
    if not query_tool:
        return "Error: sql_db_query tool not available"
    
    try:
        # Route questions to appropriate SQL queries based on keywords
        if "artist" in question.lower():
            result = query_tool.invoke("SELECT * FROM Artist LIMIT 5;")
            return f"Question: {question}\n\nResult:\n{result}"
        elif "customer" in question.lower():
            result = query_tool.invoke("SELECT COUNT(*) as count FROM Customer;")
            return f"Question: {question}\n\nResult:\n{result}"
        elif "album" in question.lower():
            result = query_tool.invoke("SELECT * FROM Album WHERE ArtistId = 1 LIMIT 5;")
            return f"Question: {question}\n\nResult:\n{result}"
        else:
            # Default query to show tables
            result = query_tool.invoke("SELECT name FROM sqlite_master WHERE type='table';")
            return f"Question: {question}\n\nAvailable tables:\n{result}"
    except Exception as e:
        return f"Error executing query: {str(e)}"

# Test the SQL Agent
if __name__ == "__main__":
    # Test 1: List all artists
    result1 = ask_sql_agent("List all artists in the database")
    print(f"Test 1:\n{result1}\n")
    
    # Test 2: Count customers
    result2 = ask_sql_agent("How many customers are there?")
    print(f"Test 2:\n{result2}\n")
    
    # Test 3: Find albums by artist
    result3 = ask_sql_agent("Show me albums by the artist with ID 1")
    print(f"Test 3:\n{result3}\n")

Test 1:
Question: List all artists in the database

Result:
[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]

Test 2:
Question: How many customers are there?

Result:
[(59,)]

Test 3:
Question: Show me albums by the artist with ID 1

Result:
[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]

